## Optimize skewed joins with broadcast joins

In [0]:
#Identify skewed product_id values by grouping sales
skewed_product_id_df = spark.sql("""
SELECT b.sale_id, COUNT(*) as sales_count
FROM silver_db.cleand_products_data a left join silver_db.cleand_sales_data b
on a.product_id = b.product_id
GROUP BY a.product_id,b.sale_id
ORDER BY b.sale_id DESC
""")
display(skewed_product_id_df)



In [0]:
#Count records per product and sort to find hot keys
/#Count how many times each product_id appears in the sales data and sort the results to find products with the highest number of records.
#The products with very high counts are called “hot keys” or skewed keys.

records_per_product = spark.sql("""select product_id, count(*)as product_record from silver_db.cleand_products_data
group by  product_id 
order by count(*)""")


display(records_per_product)


In [0]:
# Apply broadcast join to eliminate shuffle for dimension tables
# Here if we join sales table with product and stores table shuffle will get happend
from pyspark.sql.functions import broadcast
df = spark.read.table("silver_db.cleand_sales_data")
products_df = spark.read.table("silver_db.cleand_products_data")

eliminate_shuffle_df = df.join(broadcast(products_df), df.product_id == products_df.product_id, "inner")

display(eliminate_shuffle_df)

In [0]:
#Compare execution plans using .explain(True)
eliminate_shuffle_df.explain()